In [1]:
!apt-get update -qq
!apt-get install -y zstd

!pip install -q langchain langchain-core langchain-community langchain-ollama

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 47 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 1s (1,040 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122797 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
!curl -fsSL https://ollama.com/install.sh | sh

!which ollama
!ollama --version

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
/usr/local/bin/ollama


In [3]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

print("Ollama server started!")

Ollama server started!


In [4]:
!curl -s http://127.0.0.1:11434/api/tags

{"models":[]}

In [5]:
!ollama pull llama3.2:3b

In [6]:
!ollama list

NAME           ID              SIZE      MODIFIED       
llama3.2:3b    a80c4f17acd5    2.0 GB    27 seconds ago    


In [7]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import OllamaLLM

print("LangChain imports successful!")

LangChain imports successful!


In [8]:
llm = OllamaLLM(
    model="llama3.2:3b",
    temperature=0
)

print("Ollama LLM created successfully!")

Ollama LLM created successfully!


In [9]:
prompt = PromptTemplate(
    input_variables=["question"],
    template="""
You are a helpful beginner-friendly AI assistant.

Answer the user's question clearly and simply.

Question:
{question}

Answer:
"""
)

print("PromptTemplate created successfully!")

PromptTemplate created successfully!


In [10]:
output_parser = StrOutputParser()

print("OutputParser created successfully!")

OutputParser created successfully!


In [11]:
chain = prompt | llm | output_parser

print("LangChain chain created successfully!")
print("Pipeline: PromptTemplate → Ollama LLM → OutputParser")

LangChain chain created successfully!
Pipeline: PromptTemplate → Ollama LLM → OutputParser


In [12]:
questions = [
    "What is Python?",
    "What is a function in programming?",
    "What is machine learning?",
    "What is an API?",
    "What is a database?"
]

print("=" * 70)
print("LANGCHAIN CHAIN — 5 TEST INPUTS")
print("=" * 70)

for i, question in enumerate(questions, 1):
    print(f"\nQuestion {i}: {question}")

    answer = chain.invoke({
        "question": question
    })

    print("Answer:", answer)
    print("-" * 70)

LANGCHAIN CHAIN — 5 TEST INPUTS

Question 1: What is Python?
Answer: Python is a programming language that is easy to learn and use. It's often used for building websites, apps, and games, as well as for data analysis and science. Python is known for being simple, flexible, and versatile, making it a great language for beginners and experts alike!
----------------------------------------------------------------------

Question 2: What is a function in programming?
Answer: In programming, a function is a block of code that performs a specific task. It's like a recipe that you can reuse over and over again. You can call a function by its name, and it will do the same thing every time you call it.

Think of it like a phone number. When you dial a phone number, the phone will ring and connect you to the person you're trying to reach. A function is like a phone number for code - you can call it, and it will do something specific for you.

Functions are useful because they help you:

* Write

In [13]:
from langchain_core.messages import HumanMessage, AIMessage

print("Memory message classes imported successfully!")

Memory message classes imported successfully!


In [14]:
conversation_history = []

print("Conversation memory initialized!")

Conversation memory initialized!


In [15]:
memory_prompt = PromptTemplate(
    input_variables=["history", "question"],
    template="""
You are a helpful AI assistant.

Use the previous conversation when it is relevant.

Previous conversation:
{history}

New question:
{question}

Answer clearly and simply.
"""
)

print("Memory prompt created successfully!")

Memory prompt created successfully!


In [16]:
memory_chain = memory_prompt | llm | output_parser

print("Memory chain created successfully!")

Memory chain created successfully!


In [17]:
def ask_with_memory(question):
    history_text = ""

    for message in conversation_history:
        if isinstance(message, HumanMessage):
            history_text += f"User: {message.content}\n"
        elif isinstance(message, AIMessage):
            history_text += f"AI: {message.content}\n"

    answer = memory_chain.invoke({
        "history": history_text,
        "question": question
    })

    conversation_history.append(
        HumanMessage(content=question)
    )

    conversation_history.append(
        AIMessage(content=answer)
    )

    return answer

print("Memory function created successfully!")

Memory function created successfully!


In [18]:
turns = [
    "My name is Kusuma.",
    "What is Python?",
    "What can Python be used for?",
    "Which programming language did I say I am learning?",
    "Can you summarize what we discussed?"
]

print("=" * 70)
print("CONVERSATION MEMORY — 5 TURNS")
print("=" * 70)

for i, question in enumerate(turns, 1):
    print(f"\nTURN {i}")
    print("User:", question)

    answer = ask_with_memory(question)

    print("AI:", answer)
    print("-" * 70)

CONVERSATION MEMORY — 5 TURNS

TURN 1
User: My name is Kusuma.
AI: Hello Kusuma! It's nice to meet you. This is the beginning of our conversation, so we haven't discussed anything yet. How can I assist you today?
----------------------------------------------------------------------

TURN 2
User: What is Python?
AI: Hello Kusuma! I'm glad we're starting our conversation.

Python is a popular programming language used for developing various types of applications, such as websites, mobile apps, and games. It's known for its simplicity, readability, and ease of use, making it a great language for beginners and experienced programmers alike.

Would you like to know more about Python or is there something specific you'd like to learn about it?
----------------------------------------------------------------------

TURN 3
User: What can Python be used for?
AI: Hello Kusuma! I'm glad we're continuing our conversation.

Python is a versatile programming language that can be used for a wide ran

In [19]:
print("=" * 70)
print("STORED CONVERSATION HISTORY")
print("=" * 70)

for i, message in enumerate(conversation_history, 1):
    print(f"{i}. {type(message).__name__}:")
    print(message.content)
    print("-" * 70)

STORED CONVERSATION HISTORY
1. HumanMessage:
My name is Kusuma.
----------------------------------------------------------------------
2. AIMessage:
Hello Kusuma! It's nice to meet you. This is the beginning of our conversation, so we haven't discussed anything yet. How can I assist you today?
----------------------------------------------------------------------
3. HumanMessage:
What is Python?
----------------------------------------------------------------------
4. AIMessage:
Hello Kusuma! I'm glad we're starting our conversation.

Python is a popular programming language used for developing various types of applications, such as websites, mobile apps, and games. It's known for its simplicity, readability, and ease of use, making it a great language for beginners and experienced programmers alike.

Would you like to know more about Python or is there something specific you'd like to learn about it?
----------------------------------------------------------------------
5. HumanMessag

In [20]:
from langchain_core.tools import tool

@tool
def web_search_stub(query: str) -> str:
    """Simulates a web search and returns a sample result."""
    return f"Simulated web search result for: {query}"


@tool
def calculator(expression: str) -> str:
    """Calculates a basic mathematical expression."""
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return str(result)
    except Exception:
        return "Invalid mathematical expression."


print("Two tools created successfully!")
print("Tool 1:", web_search_stub.name)
print("Tool 2:", calculator.name)

Two tools created successfully!
Tool 1: web_search_stub
Tool 2: calculator


In [21]:
def simple_agent(task):
    task_lower = task.lower()

    # Use calculator for mathematical tasks
    if (
        "calculate" in task_lower
        or "solve" in task_lower
        or "+" in task_lower
        or "-" in task_lower
        or "*" in task_lower
        or "/" in task_lower
        or "%" in task_lower
    ):
        expression = task_lower

        for word in ["calculate", "solve", "what is"]:
            expression = expression.replace(word, "").strip()

        return calculator.invoke(expression)

    # Otherwise use the web search stub
    return web_search_stub.invoke(task)


print("Simple two-tool agent created successfully!")

Simple two-tool agent created successfully!


In [22]:
agent_tasks = [
    "Calculate 25 * 4",
    "Calculate 100 / 5",
    "Search for information about LangChain"
]

print("=" * 70)
print("TWO-TOOL AGENT — 3 TEST TASKS")
print("=" * 70)

for i, task in enumerate(agent_tasks, 1):
    print(f"\nTask {i}: {task}")

    result = simple_agent(task)

    print("Result:", result)
    print("-" * 70)

TWO-TOOL AGENT — 3 TEST TASKS

Task 1: Calculate 25 * 4
Result: 100
----------------------------------------------------------------------

Task 2: Calculate 100 / 5
Result: 20.0
----------------------------------------------------------------------

Task 3: Search for information about LangChain
Result: Simulated web search result for: Search for information about LangChain
----------------------------------------------------------------------


In [23]:
checklist = {
    "PromptTemplate created": True,
    "Ollama LLM created": True,
    "OutputParser created": True,
    "LangChain chain created": True,
    "5 chain inputs tested": True,
    "Conversation memory implemented": True,
    "5 conversation turns tested": True,
    "Conversation history verified": len(conversation_history) >= 10,
    "Web search stub created": True,
    "Calculator tool created": True,
    "Two-tool agent created": True,
    "3 agent tasks tested": True
}

print("=" * 70)
print("W6D5 PROJECT SELF-REVIEW")
print("=" * 70)

for item, status in checklist.items():
    symbol = "✅" if status else "❌"
    print(f"{symbol} {item}")

print("=" * 70)

if all(checklist.values()):
    print("🎉 W6D5 PROJECT COMPLETED SUCCESSFULLY!")
else:
    print("⚠️ Please review the failed items.")

W6D5 PROJECT SELF-REVIEW
✅ PromptTemplate created
✅ Ollama LLM created
✅ OutputParser created
✅ LangChain chain created
✅ 5 chain inputs tested
✅ Conversation memory implemented
✅ 5 conversation turns tested
✅ Conversation history verified
✅ Web search stub created
✅ Calculator tool created
✅ Two-tool agent created
✅ 3 agent tasks tested
🎉 W6D5 PROJECT COMPLETED SUCCESSFULLY!
